# Spring NN Exact Line-Search Experiment

This notebook trains `NNPotentialFF` with `ExactLineSearchDirectionalImplicitMultipleSystemOptimizer`. It is intentionally separate from `spring_nn_training`, because this optimizer uses force, Hessian, and line-search computations and is much slower than direct energy-label NN training.


In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import copy
import torch
import numpy as np

from geomatry.opt_Energy_approximated.optimizer import ExactLineSearchDirectionalImplicitMultipleSystemOptimizer
from geomatry.opt.loss import rmsd_loss
from geomatry.ff.graph import get_given_graph_builder
from geomatry.ff.springs import SpringFF, _random_spring_systems, _random_spring_ff_param
from geomatry.ff.NN import NNPotentialFF


In [2]:
def min_pair_distance(Ra: torch.Tensor) -> float:
    Ra = Ra.detach()
    dmat = torch.cdist(Ra, Ra)
    return float(torch.min(dmat + torch.eye(Ra.shape[0], device=Ra.device) * 1e6))

# Report a sample geometry distance to keep the setup comparable to the LJ and spring demos.


In [3]:
systemN = 20
N = 8
N_pairs = 12
max_Za = 3
fixed_atom_indices = None

# Build target spring systems and parameters.
Ras, Zas, idx_is, idx_js = _random_spring_systems(systemN, N, N_pairs, max_Za, start_Za=1)
k, r0 = _random_spring_ff_param(max_Za, k_max=2.0, r0_max=1.5)

target_ff = SpringFF(max_Za)
target_ff.reset_parameters(k, r0)
params_star = copy.deepcopy(target_ff.state_dict())

graph_builders = [get_given_graph_builder(idx_i, idx_j) for idx_i, idx_j in zip(idx_is, idx_js)]

print('sample target k', k[1:, 1:].mean().item())
print('sample target r0', r0[1:, 1:].mean().item())
print('sample min distances', [min_pair_distance(Ra) for Ra in Ras[:3]])


sample target k 1.2677267056513928
sample target r0 0.5787213469106226
sample min distances [0.22633433908102618, 0.5651950611247865, 0.7917921141725847]


In [4]:
# Use SpringFF only to generate target optimized geometries.
target_optimizer = ExactLineSearchDirectionalImplicitMultipleSystemOptimizer(
    Ras, Zas, graph_builders, target_ff, rmsd_loss,
    params_star=params_star,
    fixed_atom_indices=fixed_atom_indices,
    fmax=1e-5
)
Ra_star_list = target_optimizer.Ra_stars

# Train the NN force field with the exact line-search directional implicit optimizer.
ff = NNPotentialFF(max_Za, hidden_dim=24)
optimizer = ExactLineSearchDirectionalImplicitMultipleSystemOptimizer(
    Ras, Zas, graph_builders, ff, rmsd_loss,
    Ra_stars=Ra_star_list,
    fixed_atom_indices=fixed_atom_indices,
    fmax=1e-5
)
params_0 = copy.deepcopy(ff.state_dict())

print('target geometries', len(Ra_star_list))
print('NN parameter tensors', len(params_0))


target geometries 20
NN parameter tensors 60


In [5]:
new_params = optimizer.optimize(params_0, lr=1e-3, epoch=15001, print_period=5000, e_size=0.2, eN=9, sg=2)
print('optimization complete')


  0%|          | 1/15001 [00:00<1:26:39,  2.88it/s, alpha_star=7.5940e+01, min_epsilon=N/A, total_scale=2.062635e+08]

Iteration 0, alpha_star=7.5940e+01, total_scale=2.062635e+08


 33%|███▎      | 5001/15001 [24:43<41:49,  3.99it/s, alpha_star=4.9995e+00, min_epsilon=N/A, total_scale=8.151474e+02]  

Iteration 5000, alpha_star=4.9995e+00, total_scale=8.151474e+02


 60%|█████▉    | 8962/15001 [43:07<29:03,  3.46it/s, alpha_star=4.9995e+00, min_epsilon=N/A, total_scale=7.497245e+02]  


KeyboardInterrupt: 

In [ ]:
ff.load_state_dict(new_params)

losses = []
for Ra_star, Za, graph_builder in zip(Ra_star_list, Zas, graph_builders):
    Ra_model = optimizer.get_Ra_star(graph_builder, Ra_star, Za, new_params)
    losses.append(rmsd_loss(Ra_star, Ra_model).item())

print('mean geometry rmsd', np.mean(losses))
print('max geometry rmsd', np.max(losses))
